In [1]:
from taxonomy import ALL_TOPICS, detect_topic

In [61]:
from pathlib import Path

BASE_DIR   = Path.cwd().parent
CHROMA_DIR = BASE_DIR / "data" / "chroma_db"

COLLECTION_DESCRIPTIONS = {
    "babyos_development": (
        "fetal size and weight by week, organ development milestones, baby movements, "
        "what the baby looks like this week, heartbeat, limb development, brain growth, "
        "toddler motor milestones, baby growth charts, first steps, first words"
    ),
    "babyos_medical": (
        "danger signs requiring hospital, specific symptoms (bleeding, pain, swelling, "
        "headache, reduced movement), safe and unsafe foods, medication safety, "
        "blood test interpretation, GDM, preeclampsia, anaemia, UTI, Group B Strep"
    ),
    "babyos_postpartum": (
        "fourth trimester, newborn care, breastfeeding, formula feeding, baby sleep, "
        "postnatal depression, postpartum recovery, weaning, starting solids, "
        "separation anxiety, toddler behaviour, Kita start, return to work"
    ),
    "babyos_germany": (
        "Mutterpass, Vorsorgeuntersuchungen, Hebamme, Elterngeld, Kindergeld, "
        "Krankenkasse, U-Untersuchungen, Kita registration, Standesamt, "
        "German medical vocabulary, hospital registration in Germany, BZgA"
    ),
    "babyos_dad": (
        "dad and partner support, what to do each trimester, hospital bag checklist, "
        "how to support during labour, paternal postnatal depression, "
        "bonding with baby, practical tasks, paternity leave"
    ),
    "babyos_faqs": (
        "common questions: exercise in pregnancy, sex in pregnancy, travel, "
        "weight gain, when to tell people, miscarriage causes, movement timeline, "
        "anatomy scan, Braxton Hicks, mucus plug, waters breaking, birth plan"
    ),
    "babyos_web": (
        "NHS and WHO clinical guidelines, evidence-based recommendations, "
        "antenatal care schedule, newborn screening, postnatal checks, "
        "labour and birth procedures, infant feeding policy"
    ),
    "babyos_books": (
        "in-depth guidance from authoritative books and official guidelines — "
        "search here for: detailed nutrition advice, breastfeeding technique, "
        "birth positions, postpartum mental health, infant development theory, "
        "complications management, German BZgA parenting guides"
    ),
}
 
BOOK_TOPIC_KEYWORDS: dict[str, list[str]] = {
    "nutrition":       ["food", "eat", "diet", "nutrient", "vitamin", "iron", "folic", "supplement",
                        "calcium", "omega", "dha", "weight gain", "calorie", "safe to eat"],
    "labour":          ["labour", "labor", "birth", "contractions", "pushing", "delivery",
                        "epidural", "pain relief", "waters", "crowning", "episiotomy"],
    "newborn":         ["newborn", "baby", "nappy", "diaper", "cord", "jaundice", "APGAR",
                        "umbilical", "fontanelle", "reflex", "cry", "sleep baby"],
    "breastfeeding":   ["breastfeed", "breast feed", "latch", "milk supply", "engorgement",
                        "nipple", "colostrum", "formula", "weaning", "bottle"],
    "development":     ["milestone", "development", "crawl", "walk", "talk", "language",
                        "motor", "cogniti", "toddler", "growth"],
    "mental_health":   ["depression", "anxiety", "PND", "postnatal", "postpartum", "stress",
                        "mood", "baby blues", "identity", "matrescence"],
    "complications":   ["complication", "preeclampsia", "gestational diabetes", "preterm",
                        "miscarriage", "ectopic", "placenta", "IUGR", "stillbirth"],
    "germany":         ["mutterpass", "hebamme", "elterngeld", "vorsorge", "kita",
                        "krankenkasse", "german", "bzga", "kindergeld"],
    "general":         [],   # fallback — no filter
}
 
AGENT_COLLECTION_GUARANTEES: dict[str, list[str]] = {
    "medical_agent":   ["babyos_medical", "babyos_web"],
    "tracker_agent":   ["babyos_development"],
    "emotional_agent": ["babyos_postpartum"],
    "parent_agent":    ["babyos_dad"],
    "hebamme_agent":   ["babyos_medical", "babyos_germany"],
    "germany_agent":   ["babyos_germany"],
    "default":         [],    # pure router decision — no forced collections
}

# ALWAYS_SEARCH = {"babyos_medical"}

In [62]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

def get_embeddings():
    return OpenAIEmbeddings(
        model="text-embedding-3-small",
        openai_api_key=os.getenv("OPENAI_API_KEY"),
    )

In [63]:
from pathlib import Path
from langchain_chroma import Chroma

EMBEDDINGS = get_embeddings()

VECTOR_STORES = {}

def load_all_collections():

    loaded = []

    for collection_name in COLLECTION_DESCRIPTIONS:

        persist_path = str(CHROMA_DIR / collection_name)

        if not Path(persist_path).exists():
            continue

        try:

            VECTOR_STORES[collection_name] = Chroma(
                collection_name=collection_name,
                embedding_function=EMBEDDINGS,
                persist_directory=persist_path,
            )

            loaded.append(collection_name)

        except Exception as e:
            print(f"[RAG] Failed loading {collection_name}: {e}")

    print(f"[RAG] Loaded: {loaded}")

    return VECTOR_STORES

In [64]:
import os

from langchain_openai import ChatOpenAI

router_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=os.getenv("OPENAI_API_KEY"),
)

def route_query(
    query,
    available_collections,
    role="mom",
    phase="",
    agent_name="default",
):

    available = {
        k: v
        for k, v in COLLECTION_DESCRIPTIONS.items()
        if k in available_collections
        and k != "babyos_books"
    }

    collection_list = "\n".join(
        f"{k}: {v}"
        for k, v in available.items()
    )

    prompt = f"""
You are a routing assistant for BabyOS.

Select 1-3 collections.

IMPORTANT:
- development = growth milestones
- medical = symptoms/safety
- germany = German system
- postpartum = after birth
- faqs = common general questions

Collections:
{collection_list}

Role:
{role}

Phase:
{phase}

Query:
{query}

Return ONLY comma-separated collection names.
"""

    result = router_llm.invoke(prompt).content.strip()

    routed = {
        c.strip()
        for c in result.split(",")
        if c.strip() in available
    }

    guarantees = AGENT_COLLECTION_GUARANTEES.get(
        agent_name,
        []
    )

    routed |= set(guarantees)

    return list(routed)

In [65]:
def detect_book_topic(query):

    q = query.lower()

    scores = {}

    for topic, keywords in BOOK_TOPIC_KEYWORDS.items():

        if topic == "general":
            continue

        score = sum(
            1 for kw in keywords
            if kw in q
        )

        if score > 0:
            scores[topic] = score

    if not scores:
        return None

    return max(scores, key=scores.get)


def retrieve_from_books(
    query,
    k_books=3,
):

    if "babyos_books" not in VECTOR_STORES:
        return []

    store = VECTOR_STORES["babyos_books"]

    topic = detect_book_topic(query)

    try:

        if topic:

            docs = store.max_marginal_relevance_search(
                query,
                k=k_books,
                fetch_k=k_books * 5,
                filter={"topic_tag": topic},
            )

            if not docs:

                docs = store.max_marginal_relevance_search(
                    query,
                    k=k_books,
                    fetch_k=k_books * 5,
                )

        else:

            docs = store.max_marginal_relevance_search(
                query,
                k=k_books,
                fetch_k=k_books * 5,
            )

        for doc in docs:

            doc.metadata["retrieved_from"] = "babyos_books"

            doc.metadata["book_topic"] = topic or "general"

        return docs

    except Exception as e:

        print(f"[RAG] Books retrieval failed: {e}")

        return []

In [66]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

def get_base_retriever(
    collection_name,
    week=None,
    k=4,
):

    store = VECTOR_STORES[collection_name]

    if collection_name == "babyos_development" and week:

        return store.as_retriever(
            search_kwargs={
                "k": k,
                "filter": {
                    "week": {
                        "$gte": max(1, week - 4),
                        "$lte": min(42, week + 4),
                    }
                }
            }
        )

    return store.as_retriever(
        search_kwargs={"k": k}
    )


def retrieve_docs(
    query,
    week=None,
    role="mom",
    phase="",
    agent_name="default",
    use_multi_query=True,
    include_books=True,
    k=4,
):

    target_cols = route_query(
        query=query,
        available_collections=VECTOR_STORES.keys(),
        role=role,
        phase=phase,
        agent_name=agent_name,
    )

    all_docs = []
    seen = set()

    for collection_name in target_cols:

        try:

            retriever = get_base_retriever(
                collection_name=collection_name,
                week=week,
                k=k,
            )

            if use_multi_query:

                mq = MultiQueryRetriever.from_llm(
                    retriever=retriever,
                    llm=llm,
                )

                docs = mq.invoke(query)

            else:

                docs = retriever.invoke(query)

            for doc in docs:

                fingerprint = hashlib.md5(
                    doc.page_content.encode()
                ).hexdigest()

                if fingerprint in seen:
                    continue

                seen.add(fingerprint)

                doc.metadata["retrieved_from"] = collection_name

                all_docs.append(doc)

        except Exception as e:

            print(f"[RAG] Retrieval failed for {collection_name}: {e}")

    # parallel books retrieval

    if include_books:

        book_docs = retrieve_from_books(query)

        for doc in book_docs:

            fingerprint = hashlib.md5(
                doc.page_content.encode()
            ).hexdigest()

            if fingerprint in seen:
                continue

            seen.add(fingerprint)

            all_docs.append(doc)

    return all_docs

In [67]:
SYSTEM_PROMPT = """
You are BabyOS.

You are an educational pregnancy support assistant.

You are NOT a doctor.

Use:
- retrieved context
- pregnancy week
- user profile

If symptoms sound dangerous:
recommend contacting healthcare professionals.
"""

In [68]:
def debug_retrieve(
    query,
    week=None,
    role="mom",
    phase="",
    agent_name="default",
):

    print("=" * 60)

    print(f"QUERY : {query}")
    print(f"WEEK  : {week}")
    print(f"ROLE  : {role}")
    print(f"PHASE : {phase}")
    print(f"AGENT : {agent_name}")

    routed = route_query(
        query=query,
        available_collections=[],
        role=role,
        phase=phase,
        agent_name=agent_name,
    )

    print(f"\nRouter selected: {routed}")

    topic = detect_book_topic(query)

    print(f"Book topic: {topic}")

    docs = retrieve_docs(
        query=query,
        week=week,
        role=role,
        phase=phase,
        agent_name=agent_name,
    )

    print(f"\nRetrieved {len(docs)} docs\n")

    for doc in docs[:5]:

        print(
            f"[{doc.metadata.get('retrieved_from')}] "
            f"{doc.page_content[:120]}"
        )

        print()

In [69]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", """
Question:
{question}

Pregnancy week:
{week}

Retrieved Context:
{context}
""")
])

# def format_docs(docs):

#     return "\n\n".join(
#         d.page_content
#         for d in docs
#     )

def format_docs(docs):

    if not docs:
        return "No relevant information found."

    sections = []

    for i, doc in enumerate(docs, 1):

        source = doc.metadata.get(
            "source_file",
            "unknown"
        )

        col = doc.metadata.get(
            "retrieved_from",
            ""
        )

        topic = doc.metadata.get(
            "book_topic",
            ""
        )

        label = f"{col}/{topic}" if topic else col

        sections.append(
            f"[Source {i} — {source} ({label})]:\n"
            f"{doc.page_content}"
        )

    return "\n\n---\n\n".join(sections)

def ask_babyos(
    question,
    week=None,
    role="mom",
):

    docs = debug_retrieve(
        query=question,
        week=week,
        role=role,
    )

    context = format_docs(docs)

    chain = (
        prompt
        | llm
        | StrOutputParser()
    )

    return chain.invoke({
        "question": question,
        "week": week,
        "context": context,
    })

In [56]:
import hashlib 

In [70]:
load_vectorstores()

c:\Ironhack\W8\BabyOS\data\chroma_db\babyos_development
c:\Ironhack\W8\BabyOS\data\chroma_db\babyos_medical
c:\Ironhack\W8\BabyOS\data\chroma_db\babyos_postpartum
c:\Ironhack\W8\BabyOS\data\chroma_db\babyos_germany
c:\Ironhack\W8\BabyOS\data\chroma_db\babyos_dad
c:\Ironhack\W8\BabyOS\data\chroma_db\babyos_faqs
c:\Ironhack\W8\BabyOS\data\chroma_db\babyos_web
c:\Ironhack\W8\BabyOS\data\chroma_db\babyos_books
Loaded collections: ['babyos_development', 'babyos_medical', 'babyos_germany', 'babyos_dad', 'babyos_faqs', 'babyos_web', 'babyos_books']


{'babyos_development': <langchain_chroma.vectorstores.Chroma at 0x293d07bd880>,
 'babyos_medical': <langchain_chroma.vectorstores.Chroma at 0x293d0823e50>,
 'babyos_germany': <langchain_chroma.vectorstores.Chroma at 0x293d0889a50>,
 'babyos_dad': <langchain_chroma.vectorstores.Chroma at 0x293d2a1c2d0>,
 'babyos_faqs': <langchain_chroma.vectorstores.Chroma at 0x293d2a1d9d0>,
 'babyos_web': <langchain_chroma.vectorstores.Chroma at 0x293d2a1f0d0>,
 'babyos_books': <langchain_chroma.vectorstores.Chroma at 0x293d2a20a50>}

In [71]:
response = ask_babyos(
    question="What should dads prepare during week 30?",
    week=30,
    role="dad",
)

print(response)

QUERY : What should dads prepare during week 30?
WEEK  : 30
ROLE  : dad
PHASE : 
AGENT : default

Router selected: []
Book topic: None
[RAG] Retrieval failed for babyos_development: Expected operator expression to have exactly one operator, got {'$gte': 26, '$lte': 34} in query.

Retrieved 15 docs

[babyos_dad] ### Practical tasks for you this trimester:
- [ ] Attend the 20-week scan together
- [ ] Start the Geburtsvorbereitungsk

[babyos_dad] ---

## Third Trimester (Weeks 28–40): Preparation Mode

### What she is going through:
- Significant physical discomfor

[babyos_dad] ### What you can do:
- Go to the 20-week anatomy scan — this is the most detailed ultrasound and a major milestone
- You

[babyos_dad] ### Practical tasks for you this trimester:
- [ ] Pack the hospital bag (both yours and hers)
- [ ] Finalise the birth p

[babyos_dad] # Dad & Partner Complete Pregnancy Guide

## Your Role During Pregnancy

Being a supportive partner during pregnancy is 

During week 30 of pregnan

role = dad/mom
week = 1-40

In [72]:
response = ask_babyos(
    question="what kind of foods can i eat",
    week=23,
    role="mom",
)

print(response)

QUERY : what kind of foods can i eat
WEEK  : 23
ROLE  : mom
PHASE : 
AGENT : default

Router selected: []
Book topic: nutrition

Retrieved 13 docs

[babyos_medical] ### Foods SAFE that are often wrongly avoided:
- Well-cooked eggs
- Well-cooked fish (salmon, cod, haddock, sardines — e

[babyos_medical] ### Particularly BENEFICIAL foods:
- Leafy greens (spinach, kale) — iron, folate, calcium
- Legumes (lentils, chickpeas,

[babyos_medical] ---

## Safe and Unsafe Foods in Pregnancy

### Foods to AVOID completely:
- Raw or undercooked meat (steak tartare, car

[babyos_medical] ---

## Supplement Guide

| Supplement | Dose | Why | Start |
|-----------|------|-----|-------|
| Folic acid | 400mcg/d

[babyos_faqs] **Q: Is it normal to not feel pregnant?**
A: Yes, especially in the early weeks and during the second trimester for some

At 23 weeks pregnant, it's important to focus on a balanced diet that supports both your health and your baby's development. Here are some food groups and examp

In [73]:
response = ask_babyos(
    question="How can i support my partner",
    week=30,
    role="dad",
)

print(response)

QUERY : How can i support my partner
WEEK  : 30
ROLE  : dad
PHASE : 
AGENT : default

Router selected: []
Book topic: None

Retrieved 7 docs

[babyos_dad] # Dad & Partner Complete Pregnancy Guide

## Your Role During Pregnancy

Being a supportive partner during pregnancy is 

[babyos_dad] ### What you can do:
- Take over cooking, or at minimum keep strong cooking smells out of the house (use the extractor f

[babyos_dad] ### The birth:
- Your role is to be present, calm, and supportive — not to manage or direct
- Advocate for her birth pre

[babyos_dad] ### What you can do:
- Do not question the nesting instinct. Support it.
- Take on all heavy lifting and physically dema

[babyos_books] her care. 
- Encourage the companion to give support to the woman during 
labour and childbirth (e.g. by rubbing her bac

Supporting your partner during pregnancy, especially at 30 weeks, is crucial as they may be experiencing physical and emotional changes. Here are some ways you can help:

1. **Emoti

In [74]:
response = ask_babyos(
    question="Is Sex safe during Pregnancy",
    week=33,
    role="mom",
)

print(response)

QUERY : Is Sex safe during Pregnancy
WEEK  : 33
ROLE  : mom
PHASE : 
AGENT : default

Router selected: []
Book topic: None

Retrieved 8 docs

[babyos_faqs] **Q: Can I have sex during pregnancy?**
A: Yes, sex is safe throughout most pregnancies. The baby is protected by the am

[babyos_faqs] **Q: When is it safe to have sex after birth?**
A: Most healthcare providers advise waiting 6 weeks after birth, which c

[babyos_faqs] **Q: Can I travel during pregnancy?**
A: Yes, the second trimester (weeks 14–28) is the safest and most comfortable time

[babyos_faqs] **Q: Can I exercise during pregnancy?**
A: Yes, for most women regular moderate exercise is beneficial throughout pregna

[babyos_faqs] **Q: When should I tell people I'm pregnant?**
A: This is entirely your choice. Many couples wait until after the 12-wee

Sex is generally considered safe during pregnancy, including at 33 weeks, as long as there are no complications or specific medical concerns. Many couples continue to have a heal

In [75]:
response = ask_babyos(
    question="Could low Bp affect the baby's growth",
    week=37,
    role="mom",
)

print(response)

QUERY : Could low Bp affect the baby's growth
WEEK  : 37
ROLE  : mom
PHASE : 
AGENT : default

Router selected: []
Book topic: newborn
[RAG] Retrieval failed for babyos_development: Expected operator expression to have exactly one operator, got {'$gte': 33, '$lte': 41} in query.

Retrieved 8 docs

[babyos_medical] **Diagnosis**: Glucose tolerance test > 92mg/dL fasting, >180mg/dL at 1 hour, or >153mg/dL at 2 hours.

**Management**:


[babyos_medical] ---

## Preeclampsia — What It Means

Preeclampsia is a serious condition affecting blood pressure and organs, typically

[babyos_medical] | Headaches | Common in first and second trimester | Severe or persistent headache in third trimester (preeclampsia risk

[babyos_medical] ### First Trimester (Weeks 1–12):
- One-sided abdominal pain with or without bleeding (possible ectopic pregnancy — emer

[babyos_medical] | Symptom | Normal | Seek Help If... |
|---------|--------|----------------|
| Nausea and vomiting | Weeks 4–14 | Unable

Low bl

In [76]:
response = ask_babyos(
    question="What causes premature birth",
    week=29,
    role="mom",
)

print(response)

QUERY : What causes premature birth
WEEK  : 29
ROLE  : mom
PHASE : 
AGENT : default

Router selected: []
Book topic: labour

Retrieved 15 docs

[babyos_medical] ---

## Preeclampsia — What It Means

Preeclampsia is a serious condition affecting blood pressure and organs, typically

[babyos_medical] | Headaches | Common in first and second trimester | Severe or persistent headache in third trimester (preeclampsia risk

[babyos_medical] **Diagnosis**: Glucose tolerance test > 92mg/dL fasting, >180mg/dL at 1 hour, or >153mg/dL at 2 hours.

**Management**:


[babyos_medical] ### Third Trimester:
- **Group B Streptococcus (GBS) swab (35–37 weeks)**: If positive, antibiotics given in labour to p

[babyos_medical] ### Third Trimester (Weeks 28–40):
- Reduced or absent fetal movement (after 28 weeks, should feel 10 movements in 2 hou

Premature birth, or preterm birth, can occur for various reasons, and while some causes are known, many are still being studied. Common factors that may contribu

In [77]:
response = ask_babyos(
    question="Could spinal problems worsen during pregnancy",
    week=36,
    role="mom",
)

print(response)

QUERY : Could spinal problems worsen during pregnancy
WEEK  : 36
ROLE  : mom
PHASE : 
AGENT : default

Router selected: []
Book topic: None

Retrieved 13 docs

[babyos_medical] | Symptom | Normal | Seek Help If... |
|---------|--------|----------------|
| Nausea and vomiting | Weeks 4–14 | Unable

[babyos_medical] | Headaches | Common in first and second trimester | Severe or persistent headache in third trimester (preeclampsia risk

[babyos_medical] ### Third Trimester (Weeks 28–40):
- Reduced or absent fetal movement (after 28 weeks, should feel 10 movements in 2 hou

[babyos_medical] ### First Trimester (Weeks 1–12):
- One-sided abdominal pain with or without bleeding (possible ectopic pregnancy — emer

[babyos_medical] ---

## Preeclampsia — What It Means

Preeclampsia is a serious condition affecting blood pressure and organs, typically

While I can't provide medical advice, it's important to note that pregnancy can bring about various physical changes, and some women may experien

In [78]:
response = ask_babyos(
    question="Could low oxygen levels permanently affect the baby",
    week=25,
    role="mom",
)

print(response)

QUERY : Could low oxygen levels permanently affect the baby
WEEK  : 25
ROLE  : mom
PHASE : 
AGENT : default

Router selected: []
Book topic: newborn

Retrieved 16 docs

[babyos_medical] | Headaches | Common in first and second trimester | Severe or persistent headache in third trimester (preeclampsia risk

[babyos_medical] ---

## Supplement Guide

| Supplement | Dose | Why | Start |
|-----------|------|-----|-------|
| Folic acid | 400mcg/d

[babyos_medical] # BabyOS Medical Reference Data

## Danger Signs — When to Go to Hospital Immediately

### Any Trimester — Call Emergenc

[babyos_medical] **Diagnosis**: Glucose tolerance test > 92mg/dL fasting, >180mg/dL at 1 hour, or >153mg/dL at 2 hours.

**Management**:


[babyos_medical] ---

## Preeclampsia — What It Means

Preeclampsia is a serious condition affecting blood pressure and organs, typically

Low oxygen levels during pregnancy can potentially affect the baby, especially if they persist for an extended period. Inadequate oxygen

In [79]:
response = ask_babyos(
    question="Can i take painkillers during prgnancy",
    week=18,
    role="mom",
)

print(response)

QUERY : Can i take painkillers during prgnancy
WEEK  : 18
ROLE  : mom
PHASE : 
AGENT : default

Router selected: []
Book topic: None

Retrieved 14 docs

[babyos_medical] ---

## Safe and Unsafe Foods in Pregnancy

### Foods to AVOID completely:
- Raw or undercooked meat (steak tartare, car

[babyos_medical] | Symptom | Normal | Seek Help If... |
|---------|--------|----------------|
| Nausea and vomiting | Weeks 4–14 | Unable

[babyos_medical] | Headaches | Common in first and second trimester | Severe or persistent headache in third trimester (preeclampsia risk

[babyos_medical] ### Third Trimester (Weeks 28–40):
- Reduced or absent fetal movement (after 28 weeks, should feel 10 movements in 2 hou

[babyos_medical] ### First Trimester (Weeks 1–12):
- One-sided abdominal pain with or without bleeding (possible ectopic pregnancy — emer

During pregnancy, it's important to be cautious with medications, including painkillers. Generally, acetaminophen (Tylenol) is considered safe for occas

In [80]:
response = ask_babyos(
    question="Can baby position affect nerve pain",
    week=29,
    role="mom",
)

print(response)

QUERY : Can baby position affect nerve pain
WEEK  : 29
ROLE  : mom
PHASE : 
AGENT : default

Router selected: []
Book topic: newborn

Retrieved 14 docs

[babyos_medical] | Symptom | Normal | Seek Help If... |
|---------|--------|----------------|
| Nausea and vomiting | Weeks 4–14 | Unable

[babyos_medical] | Headaches | Common in first and second trimester | Severe or persistent headache in third trimester (preeclampsia risk

[babyos_medical] ### Third Trimester (Weeks 28–40):
- Reduced or absent fetal movement (after 28 weeks, should feel 10 movements in 2 hou

[babyos_medical] ### First Trimester (Weeks 1–12):
- One-sided abdominal pain with or without bleeding (possible ectopic pregnancy — emer

[babyos_faqs] **Q: What does placenta praevia mean?**
A: Placenta praevia means the placenta is positioned low in the uterus, partiall

At 29 weeks pregnant, it's common for the baby's position to affect various aspects of your body, including potentially contributing to nerve pain. As the 

In [81]:
response = ask_babyos(
    question="Is physiotherapy safe during pregnancy",
    week=25,
    role="dad",
)

print(response)

QUERY : Is physiotherapy safe during pregnancy
WEEK  : 25
ROLE  : dad
PHASE : 
AGENT : default

Router selected: []
Book topic: None

Retrieved 7 docs

[babyos_faqs] **Q: Can I exercise during pregnancy?**
A: Yes, for most women regular moderate exercise is beneficial throughout pregna

[babyos_faqs] **Q: Can I have sex during pregnancy?**
A: Yes, sex is safe throughout most pregnancies. The baby is protected by the am

[babyos_faqs] **Q: Can I travel during pregnancy?**
A: Yes, the second trimester (weeks 14–28) is the safest and most comfortable time

[babyos_faqs] **Q: Is it normal to not feel pregnant?**
A: Yes, especially in the early weeks and during the second trimester for some

[babyos_books] symptoms usually comprises rest, hot or cold 
compresses, and paracetamol analgesia.
nn There is a paucity of evidence o

Physiotherapy can be safe and beneficial during pregnancy, especially for managing discomfort and preparing for labor. Many pregnant individuals find relief from back 

In [82]:
response = ask_babyos(
    question="Is L4,l5 backpain is normal in pregnancy",
    week=33,
    role="mom",
)

print(response)

QUERY : Is L4,l5 backpain is normal in pregnancy
WEEK  : 33
ROLE  : mom
PHASE : 
AGENT : default

Router selected: []
Book topic: None

Retrieved 13 docs

[babyos_medical] | Symptom | Normal | Seek Help If... |
|---------|--------|----------------|
| Nausea and vomiting | Weeks 4–14 | Unable

[babyos_medical] | Headaches | Common in first and second trimester | Severe or persistent headache in third trimester (preeclampsia risk

[babyos_medical] ### Third Trimester (Weeks 28–40):
- Reduced or absent fetal movement (after 28 weeks, should feel 10 movements in 2 hou

[babyos_medical] ### First Trimester (Weeks 1–12):
- One-sided abdominal pain with or without bleeding (possible ectopic pregnancy — emer

[babyos_faqs] **Q: Can I exercise during pregnancy?**
A: Yes, for most women regular moderate exercise is beneficial throughout pregna

Back pain, especially in the lower back around the L4 and L5 regions, is quite common during pregnancy, particularly as you reach the later stages like w

In [83]:
response = ask_babyos(
    question="Why do i have moodswings",
    week=23,
    role="mom",
)

print(response)

QUERY : Why do i have moodswings
WEEK  : 23
ROLE  : mom
PHASE : 
AGENT : default

Router selected: []
Book topic: mental_health

Retrieved 14 docs

[babyos_medical] | Symptom | Normal | Seek Help If... |
|---------|--------|----------------|
| Nausea and vomiting | Weeks 4–14 | Unable

[babyos_medical] | Headaches | Common in first and second trimester | Severe or persistent headache in third trimester (preeclampsia risk

[babyos_medical] ### Third Trimester (Weeks 28–40):
- Reduced or absent fetal movement (after 28 weeks, should feel 10 movements in 2 hou

[babyos_medical] # BabyOS Medical Reference Data

## Danger Signs — When to Go to Hospital Immediately

### Any Trimester — Call Emergenc

[babyos_medical] ### Particularly BENEFICIAL foods:
- Leafy greens (spinach, kale) — iron, folate, calcium
- Legumes (lentils, chickpeas,

Mood swings during pregnancy, especially around week 23, are quite common and can be attributed to several factors:

1. **Hormonal Changes**: Your body is un

In [84]:
response = ask_babyos(
    question="Why do i get blurry vision or nausea with low bp",
    week=20,
    role="mom",
)

print(response)

QUERY : Why do i get blurry vision or nausea with low bp
WEEK  : 20
ROLE  : mom
PHASE : 
AGENT : default

Router selected: []
Book topic: None

Retrieved 12 docs

[babyos_medical] ---

## Preeclampsia — What It Means

Preeclampsia is a serious condition affecting blood pressure and organs, typically

[babyos_medical] | Headaches | Common in first and second trimester | Severe or persistent headache in third trimester (preeclampsia risk

[babyos_medical] | Symptom | Normal | Seek Help If... |
|---------|--------|----------------|
| Nausea and vomiting | Weeks 4–14 | Unable

[babyos_medical] # BabyOS Medical Reference Data

## Danger Signs — When to Go to Hospital Immediately

### Any Trimester — Call Emergenc

[babyos_faqs] ---

## First Trimester FAQs

**Q: Is morning sickness dangerous?**
A: Typical morning sickness — nausea with or without

Blurry vision and nausea can sometimes occur with low blood pressure, especially during pregnancy. Hormonal changes, increased blood volume, and 

In [85]:
response = ask_babyos(
    question="Why do my feet and ankles swell in pregnancy",
    week=31,
    role="mom",
)

print(response)

QUERY : Why do my feet and ankles swell in pregnancy
WEEK  : 31
ROLE  : mom
PHASE : 
AGENT : default

Router selected: []
Book topic: None

Retrieved 13 docs

[babyos_medical] | Headaches | Common in first and second trimester | Severe or persistent headache in third trimester (preeclampsia risk

[babyos_medical] | Symptom | Normal | Seek Help If... |
|---------|--------|----------------|
| Nausea and vomiting | Weeks 4–14 | Unable

[babyos_medical] ---

## Preeclampsia — What It Means

Preeclampsia is a serious condition affecting blood pressure and organs, typically

[babyos_medical] ### Third Trimester (Weeks 28–40):
- Reduced or absent fetal movement (after 28 weeks, should feel 10 movements in 2 hou

[babyos_medical] ### First Trimester (Weeks 1–12):
- One-sided abdominal pain with or without bleeding (possible ectopic pregnancy — emer

Swelling in the feet and ankles during pregnancy, especially around week 31, is quite common and can be attributed to several factors:

1. **Incre

In [86]:
response = ask_babyos(
    question="Is it safe to fly during pregnancy",
    week=34,
    role="dad",
)

print(response)

QUERY : Is it safe to fly during pregnancy
WEEK  : 34
ROLE  : dad
PHASE : 
AGENT : default

Router selected: []
Book topic: None

Retrieved 8 docs

[babyos_faqs] **Q: Can I travel during pregnancy?**
A: Yes, the second trimester (weeks 14–28) is the safest and most comfortable time

[babyos_faqs] **Q: Can I exercise during pregnancy?**
A: Yes, for most women regular moderate exercise is beneficial throughout pregna

[babyos_faqs] **Q: Can I have sex during pregnancy?**
A: Yes, sex is safe throughout most pregnancies. The baby is protected by the am

[babyos_faqs] **Q: Is it normal to not feel pregnant?**
A: Yes, especially in the early weeks and during the second trimester for some

[babyos_faqs] **Q: What causes miscarriage and can it be prevented?**
A: About 10–20% of known pregnancies end in miscarriage, the maj

Flying during pregnancy is generally considered safe for most women, especially up to around 36 weeks. However, there are a few factors to consider:

1. **Consult Your Heal

In [87]:
response = ask_babyos(
    question="How much coffee is allowed in pregnancy per day",
    week=17,
    role="dad",
)

print(response)

QUERY : How much coffee is allowed in pregnancy per day
WEEK  : 17
ROLE  : dad
PHASE : 
AGENT : default

Router selected: []
Book topic: None

Retrieved 9 docs

[babyos_faqs] **Q: Can I exercise during pregnancy?**
A: Yes, for most women regular moderate exercise is beneficial throughout pregna

[babyos_faqs] **Q: Can I travel during pregnancy?**
A: Yes, the second trimester (weeks 14–28) is the safest and most comfortable time

[babyos_faqs] **Q: Can I have sex during pregnancy?**
A: Yes, sex is safe throughout most pregnancies. The baby is protected by the am

[babyos_faqs] **Q: When should I tell people I'm pregnant?**
A: This is entirely your choice. Many couples wait until after the 12-wee

[babyos_faqs] **Q: Is it normal to not feel pregnant?**
A: Yes, especially in the early weeks and during the second trimester for some

During pregnancy, it's generally recommended to limit caffeine intake. Most guidelines suggest that consuming up to 200 milligrams of caffeine per day is consi

In [88]:
response = ask_babyos(
    question="Why does vaginal discharge increase in pregnancy",
    week=24,
    role="mom",
)

print(response)

QUERY : Why does vaginal discharge increase in pregnancy
WEEK  : 24
ROLE  : mom
PHASE : 
AGENT : default

Router selected: []
Book topic: None

Retrieved 12 docs

[babyos_medical] | Symptom | Normal | Seek Help If... |
|---------|--------|----------------|
| Nausea and vomiting | Weeks 4–14 | Unable

[babyos_medical] | Headaches | Common in first and second trimester | Severe or persistent headache in third trimester (preeclampsia risk

[babyos_medical] ### First Trimester (Weeks 1–12):
- One-sided abdominal pain with or without bleeding (possible ectopic pregnancy — emer

[babyos_medical] ### Third Trimester (Weeks 28–40):
- Reduced or absent fetal movement (after 28 weeks, should feel 10 movements in 2 hou

[babyos_faqs] **Q: Can I have sex during pregnancy?**
A: Yes, sex is safe throughout most pregnancies. The baby is protected by the am

During pregnancy, it's common for vaginal discharge to increase due to hormonal changes, particularly the rise in estrogen levels. This increase 

In [89]:
response = ask_babyos(
    question="Is itching with discharge normal or dangerous",
    week=20,
    role="mom",
)

print(response)

QUERY : Is itching with discharge normal or dangerous
WEEK  : 20
ROLE  : mom
PHASE : 
AGENT : default

Router selected: []
Book topic: None

Retrieved 13 docs

[babyos_medical] | Symptom | Normal | Seek Help If... |
|---------|--------|----------------|
| Nausea and vomiting | Weeks 4–14 | Unable

[babyos_medical] # BabyOS Medical Reference Data

## Danger Signs — When to Go to Hospital Immediately

### Any Trimester — Call Emergenc

[babyos_medical] ### Third Trimester (Weeks 28–40):
- Reduced or absent fetal movement (after 28 weeks, should feel 10 movements in 2 hou

[babyos_medical] ### First Trimester (Weeks 1–12):
- One-sided abdominal pain with or without bleeding (possible ectopic pregnancy — emer

[babyos_medical] **Do NOT take**: Extra vitamin A (above prenatal vitamin levels), St John's Wort, herbal supplements without medical app

Itching with discharge during pregnancy can be concerning, especially at 20 weeks. While some itching can be normal due to skin stretching or hor

In [90]:
response = ask_babyos(
    question="Is it normal to feel short of breath during pregnancy",
    week=27,
    role="dad",
)

print(response)

QUERY : Is it normal to feel short of breath during pregnancy
WEEK  : 27
ROLE  : dad
PHASE : 
AGENT : default

Router selected: []
Book topic: nutrition

Retrieved 13 docs

[babyos_medical] | Headaches | Common in first and second trimester | Severe or persistent headache in third trimester (preeclampsia risk

[babyos_medical] | Symptom | Normal | Seek Help If... |
|---------|--------|----------------|
| Nausea and vomiting | Weeks 4–14 | Unable

[babyos_medical] ### Third Trimester (Weeks 28–40):
- Reduced or absent fetal movement (after 28 weeks, should feel 10 movements in 2 hou

[babyos_medical] ### First Trimester (Weeks 1–12):
- One-sided abdominal pain with or without bleeding (possible ectopic pregnancy — emer

[babyos_medical] ---

## Preeclampsia — What It Means

Preeclampsia is a serious condition affecting blood pressure and organs, typically

Feeling short of breath during pregnancy, especially around 27 weeks, can be quite common due to the growing uterus pressing against